# Comparison of Llama8B_ecmp and Llama8B_ecmp_2 Workloads

This notebook compares the simulation results of two different workload groups: `Llama8B_ecmp` and `Llama8B_ecmp_2`. The primary difference between these two groups is the set of simulation runs they contain. The notebook will load the data for each workload group, process it, and then generate plots and tables to compare their performance and accuracy.

In [ ]:
import os
import re
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
import plotly.io as pio
from typing import Dict, List, Optional

pio.renderers.default = "plotly_mimetype"

# --- Helper Functions ---
def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        pass
    return params

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0

### 2. Create a Reusable Data Loading Function

This function, `load_workload_data`, takes a `WORKLOAD_GROUP` name as input and processes the simulation results for that group. It navigates the directory structure, parses the relevant files, and returns a pandas DataFrame containing the consolidated data. This modular approach allows us to easily load data for different workload groups.

In [ ]:
def load_workload_data(workload_group: str) -> Optional[pd.DataFrame]:
    """
    Loads all simulation results for a given workload group into a pandas DataFrame.
    """
    BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
    TOPOLOGIES = ['experiment3/FoldedClosECMP']
    
    all_results = []

    for topo in TOPOLOGIES:
        topo_path = os.path.join(BASE_OUTPUT_DIR, topo, workload_group)
        if not os.path.isdir(topo_path):
            print(f"Directory not found for topology {topo} and workload {workload_group}, skipping.")
            continue

        for workload_name in os.listdir(topo_path):
            workload_path = os.path.join(topo_path, workload_name)
            if not os.path.isdir(workload_path):
                continue

            workload_results = {'workload': workload_name, 'topology': topo}
            g2_exec_times, g2_sim_times = [], []
            ns3_exec_times, ns3_sim_times = [], []
            
            run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]
            for run_dir_name in run_dirs:
                run_path = os.path.join(workload_path, run_dir_name)
                
                sim_type = None
                if os.path.isdir(os.path.join(run_path, 'g2')):
                    sim_type = 'g2'
                elif os.path.isdir(os.path.join(run_path, 'ns3')):
                    sim_type = 'ns3'
                elif os.path.isdir(os.path.join(run_path, 'analytical_unaware')):
                    sim_type = 'analytical_unaware'
                
                if not sim_type:
                    continue

                sim_path = os.path.join(run_path, sim_type)
                sim_name = {'g2': 'G2', 'ns3': 'NS3', 'analytical_unaware': 'Analytical'}[sim_type]

                timing_file = next((os.path.join(sim_path, f) for f in os.listdir(sim_path) if 'trace_matched_timing.csv' in f), None)
                max_time_ns = None
                if timing_file:
                    try:
                        df_timing = pd.read_csv(timing_file)
                        if 'callback_tick' in df_timing.columns:
                            max_time_ns = df_timing['callback_tick'].max()
                    except Exception as e:
                        print(f"Error reading {timing_file}: {e}")
                
                summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
                sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))

                if sim_type == 'g2':
                    if max_time_ns is not None: g2_exec_times.append(max_time_ns)
                    if sim_time_sec is not None: g2_sim_times.append(sim_time_sec)
                elif sim_type == 'ns3':
                    if max_time_ns is not None: ns3_exec_times.append(max_time_ns)
                    if sim_time_sec is not None: ns3_sim_times.append(sim_time_sec)
                else:
                    workload_results[f'{sim_name}_Est_Exec_Time_ns'] = max_time_ns
                    workload_results[f'{sim_name}_Sim_Time_sec'] = sim_time_sec
            
            if g2_exec_times:
                workload_results['G2_Est_Exec_Time_ns'] = sum(g2_exec_times) / len(g2_exec_times)
                workload_results['G2_Est_Exec_Time_ns_min'] = min(g2_exec_times)
                workload_results['G2_Est_Exec_Time_ns_max'] = max(g2_exec_times)
            if g2_sim_times:
                workload_results['G2_Sim_Time_sec'] = sum(g2_sim_times) / len(g2_sim_times)
                workload_results['G2_Sim_Time_sec_min'] = min(g2_sim_times)
                workload_results['G2_Sim_Time_sec_max'] = max(g2_sim_times)
            
            if ns3_exec_times:
                workload_results['NS3_Est_Exec_Time_ns'] = sum(ns3_exec_times) / len(ns3_exec_times)
                workload_results['NS3_Est_Exec_Time_ns_min'] = min(ns3_exec_times)
                workload_results['NS3_Est_Exec_Time_ns_max'] = max(ns3_exec_times)
            if ns3_sim_times:
                workload_results['NS3_Sim_Time_sec'] = sum(ns3_sim_times) / len(ns3_sim_times)
                workload_results['NS3_Sim_Time_sec_min'] = min(ns3_sim_times)
                workload_results['NS3_Sim_Time_sec_max'] = max(ns3_sim_times)
            
            if len(workload_results) > 2:
                 all_results.append(workload_results)

    if not all_results:
        return None

    df = pd.DataFrame(all_results)
    required_cols = ['G2_Est_Exec_Time_ns', 'NS3_Est_Exec_Time_ns', 'Analytical_Est_Exec_Time_ns']
    df.dropna(subset=required_cols, inplace=True)
    
    # Add short_workload
    df['short_workload'] = df['workload'].str.replace('Llama8B_last_', '', regex=False).str.replace('.seq_2048.batch_64', '', regex=False).str.replace('_', '-', regex=False)
    
    df.sort_values(by=['topology', 'workload'], inplace=True)
    
    return df

### 3. Load Data for Each Workload Group

Now, we'll use the `load_workload_data` function to load the results for our two target workload groups: `Llama8B_ecmp` and `Llama8B_ecmp_2`. The results will be stored in two separate DataFrames, `df_ecmp1` and `df_ecmp2`.

In [ ]:
WORKLOAD_GROUP_1 = 'Llama8B_ecmp'
WORKLOAD_GROUP_2 = 'Llama8B_ecmp_2'

df_ecmp1 = load_workload_data(WORKLOAD_GROUP_1)
df_ecmp2 = load_workload_data(WORKLOAD_GROUP_2)

if df_ecmp1 is not None:
    print(f"Successfully loaded {len(df_ecmp1)} results for {WORKLOAD_GROUP_1}")
else:
    print(f"No results found for {WORKLOAD_GROUP_1}")

if df_ecmp2 is not None:
    print(f"Successfully loaded {len(df_ecmp2)} results for {WORKLOAD_GROUP_2}")
else:
    print(f"No results found for {WORKLOAD_GROUP_2}")

### 4. Combine and Preprocess Data

To make comparison easier, we'll combine the two DataFrames into a single one. We'll add a `workload_group` column to distinguish between the two sets of results. We will also precompute error and speedup columns.

In [ ]:
if df_ecmp1 is not None and df_ecmp2 is not None:
    df_ecmp1['workload_group'] = WORKLOAD_GROUP_1
    df_ecmp2['workload_group'] = WORKLOAD_GROUP_2

    df_combined = pd.concat([df_ecmp1, df_ecmp2], ignore_index=True)

    # Precompute error and speedup columns
    df_combined['G2 Error (%)'] = ((df_combined['G2_Est_Exec_Time_ns'] - df_combined['NS3_Est_Exec_Time_ns']) / df_combined['NS3_Est_Exec_Time_ns']) * 100
    df_combined['AU Error (%)'] = ((df_combined['Analytical_Est_Exec_Time_ns'] - df_combined['NS3_Est_Exec_Time_ns']) / df_combined['NS3_Est_Exec_Time_ns']) * 100
    df_combined['G2 Speedup (x)'] = df_combined['NS3_Sim_Time_sec'] / df_combined['G2_Sim_Time_sec']
    df_combined['AU Speedup (x)'] = df_combined['NS3_Sim_Time_sec'] / df_combined['Analytical_Sim_Time_sec']
    
    print("DataFrames combined and preprocessed.")
    display(df_combined.head())
else:
    print("One or both DataFrames are missing, cannot combine.")

### 5. Compare Estimated Execution Times Across Workloads

This plot shows the absolute estimated execution times for each parallelization strategy, grouped by the workload group. This allows for a direct comparison of performance between `Llama8B_ecmp` and `Llama8B_ecmp_2`.

In [ ]:
if 'df_combined' in locals() and not df_combined.empty:
    fig = go.Figure()

    for i, workload_group in enumerate(df_combined['workload_group'].unique()):
        df_group = df_combined[df_combined['workload_group'] == workload_group]
        
        # NS3
        fig.add_trace(go.Bar(
            name=f'NS3 ({workload_group})',
            x=df_group['short_workload'],
            y=df_group['NS3_Est_Exec_Time_ns'],
            error_y=dict(
                type='data',
                symmetric=False,
                array=df_group['NS3_Est_Exec_Time_ns_max'] - df_group['NS3_Est_Exec_Time_ns'],
                arrayminus=df_group['NS3_Est_Exec_Time_ns'] - df_group['NS3_Est_Exec_Time_ns_min']
            )
        ))
        # G2
        fig.add_trace(go.Bar(
            name=f'G2 ({workload_group})',
            x=df_group['short_workload'],
            y=df_group['G2_Est_Exec_Time_ns'],
            error_y=dict(
                type='data',
                symmetric=False,
                array=df_group['G2_Est_Exec_Time_ns_max'] - df_group['G2_Est_Exec_Time_ns'],
                arrayminus=df_group['G2_Est_Exec_Time_ns'] - df_group['G2_Est_Exec_Time_ns_min']
            )
        ))
        # Analytical
        fig.add_trace(go.Bar(
            name=f'Analytical ({workload_group})',
            x=df_group['short_workload'],
            y=df_group['Analytical_Est_Exec_Time_ns']
        ))

    fig.update_layout(
        title='Comparison of Estimated Execution Times',
        xaxis_title='Workload (Parallelization Strategy)',
        yaxis_title='Estimated Execution Time (ns)',
        barmode='group',
        xaxis_tickangle=-45,
        template='plotly_white',
        font=dict(size=16)
    )
    fig.show()
else:
    print("Combined DataFrame not available for plotting.")

### 6. Compare Simulation Runtimes and Speedups

These plots compare the wall-clock simulation time and the speedup of G2 and Analytical simulators over NS3. This helps in understanding the efficiency of the different simulation models.

In [ ]:
if 'df_combined' in locals() and not df_combined.empty:
    # Plot for Simulation Time
    fig_sim_time = go.Figure()
    for workload_group in df_combined['workload_group'].unique():
        df_group = df_combined[df_combined['workload_group'] == workload_group]
        fig_sim_time.add_trace(go.Bar(name=f'NS3 ({workload_group})', x=df_group['short_workload'], y=df_group['NS3_Sim_Time_sec']))
        fig_sim_time.add_trace(go.Bar(name=f'G2 ({workload_group})', x=df_group['short_workload'], y=df_group['G2_Sim_Time_sec']))
        fig_sim_time.add_trace(go.Bar(name=f'Analytical ({workload_group})', x=df_group['short_workload'], y=df_group['Analytical_Sim_Time_sec']))

    fig_sim_time.update_layout(
        title='Comparison of Simulation Runtimes',
        xaxis_title='Workload (Parallelization Strategy)',
        yaxis_title='Simulation Time (seconds)',
        barmode='group',
        xaxis_tickangle=-45,
        template='plotly_white',
        font=dict(size=16)
    )
    fig_sim_time.show()

    # Plot for Speedup
    fig_speedup = go.Figure()
    for workload_group in df_combined['workload_group'].unique():
        df_group = df_combined[df_combined['workload_group'] == workload_group]
        fig_speedup.add_trace(go.Bar(name=f'G2 Speedup ({workload_group})', x=df_group['short_workload'], y=df_group['G2 Speedup (x)']))
        fig_speedup.add_trace(go.Bar(name=f'Analytical Speedup ({workload_group})', x=df_group['short_workload'], y=df_group['AU Speedup (x)']))

    fig_speedup.update_layout(
        title='Comparison of Simulation Speedups (vs NS3)',
        xaxis_title='Workload (Parallelization Strategy)',
        yaxis_title='Speedup (x)',
        barmode='group',
        xaxis_tickangle=-45,
        template='plotly_white',
        font=dict(size=16)
    )
    fig_speedup.show()
else:
    print("Combined DataFrame not available for plotting.")

### 7. Analyze Error and Correlation by Workload Group

Finally, we'll compute and display summary statistics, such as the Mean Absolute Percentage Error (MAPE), for G2 and Analytical models, grouped by the workload group. We'll also calculate Spearman's rank correlation to evaluate how well the faster simulators preserve the performance ranking of workloads compared to NS3.

In [ ]:
if 'df_combined' in locals() and not df_combined.empty:
    for workload_group in df_combined['workload_group'].unique():
        df_group = df_combined[df_combined['workload_group'] == workload_group]
        
        print(f"--- Analysis for {workload_group} ---")
        
        # MAPE
        mape_g2 = df_group['G2 Error (%)'].abs().mean()
        mape_au = df_group['AU Error (%)'].abs().mean()
        print(f"Mean Absolute Percentage Error (MAPE):")
        print(f"  - G2 vs NS3: {mape_g2:.2f}%")
        print(f"  - Analytical vs NS3: {mape_au:.2f}%")

        # Spearman's Rank Correlation
        g2_corr, _ = spearmanr(df_group['NS3_Est_Exec_Time_ns'], df_group['G2_Est_Exec_Time_ns'])
        au_corr, _ = spearmanr(df_group['NS3_Est_Exec_Time_ns'], df_group['Analytical_Est_Exec_Time_ns'])
        print(f"Spearman's Rank Correlation (vs NS3):")
        print(f"  - G2: {g2_corr:.4f}")
        print(f"  - Analytical: {au_corr:.4f}")
        print("\\n")

    # --- Summary Table ---
    summary_df = df_combined.groupby('workload_group').agg({
        'G2 Error (%)': lambda x: x.abs().mean(),
        'AU Error (%)': lambda x: x.abs().mean(),
        'G2 Speedup (x)': 'mean',
        'AU Speedup (x)': 'mean'
    }).rename(columns={
        'G2 Error (%)': 'G2 MAPE (%)',
        'AU Error (%)': 'AU MAPE (%)',
        'G2 Speedup (x)': 'Avg G2 Speedup (x)',
        'AU Speedup (x)': 'Avg AU Speedup (x)'
    })
    
    print("--- Overall Comparison Summary ---")
    display(summary_df.style.format({
        'G2 MAPE (%)': '{:.2f}%',
        'AU MAPE (%)': '{:.2f}%',
        'Avg G2 Speedup (x)': '{:.2f}x',
        'Avg AU Speedup (x)': '{:.2f}x'
    }))
else:
    print("Combined DataFrame not available for analysis.")

### 8. Individual Simulator Comparison Across Workload Groups

This section focuses on comparing the performance of each simulator (`NS3`, `G2`, `Analytical`) between the `Llama8B_ecmp` and `Llama8B_ecmp_2` workload groups. We'll calculate the percentage difference in estimated execution time for each workload and then find the average difference. This helps to quantify the impact of the changes between the two experimental setups on each simulator.

In [ ]:
if 'df_ecmp1' in locals() and df_ecmp1 is not None and 'df_ecmp2' in locals() and df_ecmp2 is not None:
    # Merge the two dataframes on workload to compare them
    df_compare = pd.merge(
        df_ecmp1,
        df_ecmp2,
        on='short_workload',
        suffixes=(f'_{WORKLOAD_GROUP_1}', f'_{WORKLOAD_GROUP_2}')
    )

    if not df_compare.empty:
        # Calculate the percentage difference for each simulator
        for sim in ['NS3', 'G2', 'Analytical']:
            col1 = f'{sim}_Est_Exec_Time_ns_{WORKLOAD_GROUP_1}'
            col2 = f'{sim}_Est_Exec_Time_ns_{WORKLOAD_GROUP_2}'
            df_compare[f'{sim} % Diff'] = ((df_compare[col2] - df_compare[col1]) / df_compare[col1]) * 100

        # --- Plot the percentage differences ---
        fig_diff = go.Figure()
        for sim in ['NS3', 'G2', 'Analytical']:
            fig_diff.add_trace(go.Bar(
                name=f'{sim} % Difference',
                x=df_compare['short_workload'],
                y=df_compare[f'{sim} % Diff']
            ))
        
        fig_diff.update_layout(
            title=f'Percentage Difference in Est. Exec. Time ({WORKLOAD_GROUP_2} vs {WORKLOAD_GROUP_1})',
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Percentage Difference (%)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        fig_diff.show()

        # --- Calculate and print average differences ---
        print("--- Average Percentage Difference in Estimated Execution Time (filtered) ---")
        for sim in ['NS3', 'G2', 'Analytical']:
            diff_series = df_compare[f'{sim} % Diff']
            filtered_diff = diff_series[diff_series.between(-5, 5)]
            avg_diff = filtered_diff.mean()
            print(f"  - Average {sim} Difference: {avg_diff:.2f}%")
            
    else:
        print("No common workloads found between the two groups to compare.")
else:
    print("One or both DataFrames are missing, cannot perform individual comparison.")